# 💣 Global Terrorism Data Analysis
**GTD Dataset — 181,691 incidents (1970–2017)** | Advanced EDA

---

In [ ]:
import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
print("Libraries loaded")

## 1. Load & Clean Dataset

In [ ]:
cols = ["iyear","imonth","iday","country_txt","region_txt","city",
        "latitude","longitude","attacktype1_txt","targtype1_txt",
        "gname","nkill","nwound","success"]
df = pd.read_csv("globalterrorismdb_utf8.csv", usecols=cols, low_memory=False)
df = df.rename(columns={
    "iyear":"Year","imonth":"Month","iday":"Day",
    "country_txt":"Country","region_txt":"Region","city":"City",
    "latitude":"Lat","longitude":"Lon",
    "attacktype1_txt":"Attack Type","targtype1_txt":"Target Type",
    "gname":"Group","nkill":"Killed","nwound":"Wounded","success":"Success"
})
df["Killed"]     = pd.to_numeric(df["Killed"],   errors="coerce").fillna(0)
df["Wounded"]    = pd.to_numeric(df["Wounded"],  errors="coerce").fillna(0)
df["Casualties"] = df["Killed"] + df["Wounded"]
print(f"Shape: {df.shape}")
print(f"Years: {df['Year'].min()} - {df['Year'].max()}")
print(f"Countries: {df['Country'].nunique()}")
df.head()

## 2. Summary Statistics

In [ ]:
print("=== DATASET SUMMARY ===")
print(f"Total incidents : {len(df):,}")
print(f"Total killed    : {int(df['Killed'].sum()):,}")
print(f"Total wounded   : {int(df['Wounded'].sum()):,}")
print(f"Countries       : {df['Country'].nunique()}")
print(f"Groups          : {df['Group'].nunique()}")
print(f"Success rate    : {df['Success'].mean()*100:.1f}%")
print("\nMissing values:")
print(df.isnull().sum())

## 3. Annual Trend

In [ ]:
yearly = df.groupby("Year").agg(
    Incidents=("Year","count"),
    Killed=("Killed","sum"),
    Wounded=("Wounded","sum"),
).reset_index()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["Annual Incidents","Annual Casualties"])
fig.add_trace(go.Bar(x=yearly["Year"], y=yearly["Incidents"],
    name="Incidents", marker_color="steelblue"), row=1, col=1)
fig.add_trace(go.Scatter(x=yearly["Year"], y=yearly["Killed"],
    name="Killed", line=dict(color="crimson")), row=2, col=1)
fig.add_trace(go.Scatter(x=yearly["Year"], y=yearly["Wounded"],
    name="Wounded", line=dict(color="orange")), row=2, col=1)
fig.update_layout(height=600, title="Terrorism Trends 1970–2017",
                  hovermode="x unified")
fig.show()

## 4. World Map — Incidents by Country

In [ ]:
country_stats = df.groupby("Country").agg(
    Incidents=("Year","count"),
    Killed=("Killed","sum"),
    Casualties=("Casualties","sum"),
).reset_index()

fig = px.choropleth(country_stats, locations="Country", locationmode="country names",
    color="Incidents", hover_name="Country",
    hover_data=["Killed","Casualties"],
    color_continuous_scale="YlOrRd",
    title="Total Terrorism Incidents by Country (1970–2017)")
fig.update_layout(height=520)
fig.show()

## 5. Bubble Map — Attack Locations

In [ ]:
city_df = df.dropna(subset=["Lat","Lon"]).groupby(["City","Lat","Lon"]).agg(
    Incidents=("Year","count"), Killed=("Killed","sum")
).reset_index()
fig = px.scatter_geo(city_df, lat="Lat", lon="Lon",
    color="Killed", size="Incidents", hover_name="City",
    color_continuous_scale="Reds", projection="natural earth",
    title="Attack Locations (size = incidents, color = killed)")
fig.update_layout(height=500)
fig.show()

## 6. Top Groups & Countries

In [ ]:
top_groups = (df[df["Group"]!="Unknown"]
              .groupby("Group").agg(Incidents=("Year","count"), Killed=("Killed","sum"))
              .nlargest(15,"Incidents").reset_index())
fig = px.bar(top_groups, x="Incidents", y="Group", orientation="h",
             color="Killed", color_continuous_scale="Reds",
             title="Top 15 Most Active Terrorist Groups")
fig.update_layout(height=520, yaxis=dict(autorange="reversed"))
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=2, specs=[[{"type":"pie"},{"type":"pie"}]])
at = df["Attack Type"].value_counts().head(8)
tt = df["Target Type"].value_counts().head(8)
fig.add_trace(go.Pie(labels=at.index, values=at.values, name="Attack", hole=0.4), row=1, col=1)
fig.add_trace(go.Pie(labels=tt.index, values=tt.values, name="Target", hole=0.4), row=1, col=2)
fig.update_layout(title="Attack Types vs Target Types", height=420)
fig.show()

## 7. Region × Attack Type Heatmap

In [ ]:
pivot = df.groupby(["Region","Attack Type"]).size().unstack(fill_value=0)
fig = px.imshow(pivot, color_continuous_scale="YlOrRd", text_auto=True,
                title="Incidents: Region × Attack Type", aspect="auto")
fig.update_layout(height=520)
fig.show()

## 8. Region Trend Over Time

In [ ]:
region_year = df.groupby(["Year","Region"]).size().reset_index(name="Incidents")
fig = px.area(region_year, x="Year", y="Incidents", color="Region",
              title="Incidents by Region Over Time (1970–2017)")
fig.update_layout(hovermode="x unified", height=450)
fig.show()

## 9. Deadliest Incidents

In [ ]:
worst = df.nlargest(20,"Killed")[
    ["Year","Country","City","Attack Type","Group","Killed","Wounded"]
].reset_index(drop=True)
worst

## 10. Key Insights

In [ ]:
print("=== TERRORISM KEY INSIGHTS ===")
print(f"Total incidents     : {len(df):,}")
print(f"Total killed        : {int(df['Killed'].sum()):,}")
print(f"Most active group   : {df[df['Group']!='Unknown']['Group'].value_counts().index[0]}")
print(f"Most affected       : {country_stats.nlargest(1,'Incidents')['Country'].values[0]}")
peak = yearly.nlargest(1,"Incidents")
print(f"Peak year           : {int(peak['Year'].values[0])} ({int(peak['Incidents'].values[0]):,} incidents)")
print(f"Most common attack  : {df['Attack Type'].mode()[0]}")
print(f"Most targeted       : {df['Target Type'].mode()[0]}")